# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


## Again - please note: pro-tip for using Colab:

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [4]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Constants

LLAMA = "google/gemma4:26b-mlx"

In [5]:
# Check if the file is in current directory or parent directory
if os.path.exists("denver_extract.mp3"):
    audio_filename = "denver_extract.mp3"
else:
    audio_filename = "../denver_extract.mp3"
print("Audio file found:", os.path.exists(audio_filename))


Audio file found: True


# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [6]:
# Sign in to HuggingFace Hub using your environment variable
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
else:
    print("HF_TOKEN not found in environment. Proceeding with public access.")
# Open the file for API usage (optional for local pipeline)
audio_file = open(audio_filename, "rb")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# STEP 1: Transcribe Audio

## Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [11]:
from transformers import pipeline
import torch

# Select MPS for Apple Silicon GPU, or fallback to CPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small.en",
    dtype=torch.float32,
    device=device,
    return_timestamps=True
)
result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

Using device: mps


Device set to use mps


 kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world there's a lot of situations where water is very important right now and it's a very big issue so that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and Kind of share this whole idea of indigenous people's day. So, thank you. Thank you so much and thanks for your leadership All right Welcome to the Denver City Council meeting of Monday, October 9th, please rise with the Pledge of Allegiance by Councilman Lopez I pledge allegiance to the flag of the United St

In [12]:
open_source_transcription = transcription

## Option 2: Use OpenAI for Transcription

In [ ]:
# Sign in to OpenAI using Secrets in Colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# STEP 2: Analyze & Report

In [13]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [14]:
from openai import OpenAI

# Connect to your local Ollama instance
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Ollama does not need a real key, but requires any non-empty string
)

# Choose one of your local Ollama models (e.g., 'gemma4:e4b-mlx')
model_name = "gemma4:e4b-mlx"

print(f"Generating minutes with {model_name}...")

# Call Ollama with streaming enabled
stream = client.chat.completions.create(
    model=model_name,
    messages=messages,
    stream=True
)

full_response = ""
for chunk in stream:
    token = chunk.choices[0].delta.content or ""
    full_response += token
    print(token, end="", flush=True)


Generating minutes with gemma4:e4b-mlx...
# Denver City Council Meeting Minutes

## Summary

**Date:** Monday, October 9th
**Location:** Denver City Council Chambers
**Attendees:** Councilmembers Lopez, Rocco, Black, Espinosa, Flynn, Gilmour, Cashman, Kaneech, New, Ortega, Sussman, Clark, Martíga, and Kniech (11 members present).
**Meeting Focus:** The primary focus of the meeting was the observance and dedication of Proclamation 1127, recognizing Indigenous Peoples' Day in the city and county of Denver.

## Key Discussion Points

*   **Proclamation 1127 (Indigenous Peoples' Day):** The Council officially read and discussed the proclamation recognizing the ancestral homelands of numerous Indigenous tribes, including the Southern Ute Mountain Ute Tribes, Arapaho, and Cheyenne.
*   **Theme of Confluence:** The discussion highlighted the symbolism of "confluence" (the merging of rivers), relating it to the cultural confluence represented by Indigenous communities building the city of Denv

In [ ]:
response = tokenizer.decode(outputs[0])

In [15]:
display(Markdown(full_response))

# Denver City Council Meeting Minutes

## Summary

**Date:** Monday, October 9th
**Location:** Denver City Council Chambers
**Attendees:** Councilmembers Lopez, Rocco, Black, Espinosa, Flynn, Gilmour, Cashman, Kaneech, New, Ortega, Sussman, Clark, Martíga, and Kniech (11 members present).
**Meeting Focus:** The primary focus of the meeting was the observance and dedication of Proclamation 1127, recognizing Indigenous Peoples' Day in the city and county of Denver.

## Key Discussion Points

*   **Proclamation 1127 (Indigenous Peoples' Day):** The Council officially read and discussed the proclamation recognizing the ancestral homelands of numerous Indigenous tribes, including the Southern Ute Mountain Ute Tribes, Arapaho, and Cheyenne.
*   **Theme of Confluence:** The discussion highlighted the symbolism of "confluence" (the merging of rivers), relating it to the cultural confluence represented by Indigenous communities building the city of Denver.
*   **Cultural and Historical Context:** Council members emphasized that the city's history is built upon these ancestral homelands and thanked the Indigenous community for their vast contributions to local knowledge, science, and culture.
*   **Confluence Week:** The purpose of Confluence Week was discussed as a method to highlight and bring attention to local Indigenous events and contributions.
*   **Commitment to Action:** Remarks by various council members stressed that Indigenous Peoples Day is not merely a day of celebration, but a "day on"—a commitment to addressing ongoing issues such as poverty, lack of access to services (housing, sobriety, employment), and cultural challenges faced by Indigenous populations today.
*   **Environmental Intersection:** Council commentary noted the vital intersection between cultural preservation and environmental protection, specifically mentioning Indigenous communities' role in defending public, sacred lands.
*   **Inclusivity and Respect:** Several participants emphasized that the celebration must be rooted in genuine inclusivity and respect, ensuring it is not an act of replacement or disrespect toward any culture.

## Takeaways

*   **Official Recognition:** Proclamation 1127 was officially adopted, formally marking October 9, 2017, as Indigenous Peoples Day for the City and County of Denver.
*   **Reframing the Celebration:** The Council consistently framed the observance of Indigenous Peoples' Day as a dual commitment: celebrating historical contributions while actively fighting contemporary issues and promoting equity and inclusion.
*   **Power of Imagery:** The symbolic power of the event's artwork and imagery was recognized as a critical component in conveying the depth of Indigenous history and dedication.

## Action Items

| Action Item | Owner(s) | Status |
| :--- | :--- | :--- |
| **Proclamation Transmission** | Clerk of the City and County of Denver | Complete (To transmit copies to the Denver American Indian Commission, Denver School District #1, and the Colorado Commission on Indian Affairs.) |
| **Advocacy/Action** | Council Members (as a collective) | Ongoing (Continue to address critical issues facing the Indigenous community, such as poverty, housing, and access to services.) |

# Student contribution

Student Emad S. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D